## Import libraries and modules

In [ ]:
import pandas as pd
import numpy as np
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import joblib

## Dataset extraction

In [ ]:
df_generation = pd.read_csv('../data/raw/Generacion.csv')
df_demand = pd.read_csv('../data/raw/Demanda.csv')
df_capacity = pd.read_csv('../data/Capacidad.csv')

Raw data visualization

In [ ]:
df_generation = df_generation[['Fecha','GeneracionRealEstimada']]
df_generation.head()

In [ ]:
df_capacity.head()

In [ ]:
df_demand.head()

## Functions

In [ ]:
def transform_dataset(df, date_col, value_col, source_unit):
    # Convert date to YYYY-MM-DD format
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col]).dt.strftime('%Y-%m-%d')
    df.rename(columns={date_col: 'Fecha', value_col: 'Valor'}, inplace=True)
    
    # Convert to GWh based on source unit
    if source_unit.lower() == 'kwh':
        df['Valor'] = df['Valor'] / 1e6
    elif source_unit.lower() == 'mwh':
        df['Valor'] = df['Valor'] / 1e3
    # If already in GWh, no transformation needed
    
    # Keep only required columns
    df = df[['Fecha', 'Valor']]
    return df

def sum_daily(df):
    df_grouped = df.groupby('Fecha', as_index=False)['Valor'].sum()
    return df_grouped

## Prophet model

from prophet.diagnostics import cross_validation, performance_metrics

def train_predict_validate_prophet(df_daily, variable_name, end_date="2025-10-31"):
    # Data preparation and model fitting
    df_prophet = df_daily.rename(columns={'Fecha': 'ds', 'Valor': 'y'})
    df_prophet['ds'] = pd.to_datetime(df_prophet['ds'])
    df_prophet = df_prophet[df_prophet['ds'] <= end_date]
    
    def get_colombia_holidays(year):
        return pd.DataFrame({
            'holiday': 'colombia_holiday',
            'ds': [pd.to_datetime(f'{year}-01-01'), pd.to_datetime(f'{year}-07-20'), pd.to_datetime(f'{year}-12-25')],
            'lower_window': 0,
            'upper_window': 1
        })
    years = range(df_prophet['ds'].min().year, pd.to_datetime(end_date).year + 1)
    holidays_df = pd.concat([get_colombia_holidays(a) for a in years]
                            ).drop_duplicates(subset=['ds']).reset_index(drop=True)

    model = Prophet(
        holidays=holidays_df,
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False,
        seasonality_mode='multiplicative'
    )
    model.fit(df_prophet)
    
    # Prophet cross-validation
    df_cv = cross_validation(
        model, 
        initial='730 days',
        period='180 days',
        horizon = '365 days'
    )
    df_p = performance_metrics(df_cv)
    print(f"\n--- {variable_name}: Cross-Validation Metrics ---")
    print(df_p[['horizon', 'mae', 'rmse', 'mape', 'smape']].groupby('horizon').mean())

    if 'y' in df_cv and 'yhat' in df_cv:
        r2 = r2_score(df_cv['y'], df_cv['yhat'])
        print(f'Global R2 (manual, cross-validation): {r2:.4f}')

    # Extended prediction through 2030
    pred_end_date = pd.to_datetime("2030-12-31")
    future_days = (pred_end_date - df_prophet['ds'].max()).days
    future = model.make_future_dataframe(periods=future_days)
    forecast = model.predict(future)

    # Actual vs. predicted comparison and chart
    df_pred = pd.merge(df_prophet, forecast[['ds', 'yhat']], on='ds', how='inner')
    plt.figure(figsize=(14,6))
    plt.plot(df_prophet['ds'], df_prophet['y'], label=f'{variable_name} Actual')
    plt.plot(forecast['ds'], forecast['yhat'], label=f'Prophet Prediction ({variable_name})')
    plt.axvline(pd.to_datetime(end_date), color='k', linestyle='--', alpha=0.5, label='End of actual data')
    plt.title(f'{variable_name}: Actual vs. Predicted (with Cross-Validation)')
    plt.xlabel('Date')
    plt.ylabel('Daily GWh')
    plt.legend()
    plt.tight_layout()
    plt.show()
    return model, forecast, df_cv, df_p

## Dataset transformation to Date | Value (GWh)

In [ ]:
df_generation_std = transform_dataset(df_generation, 'Fecha', 'GeneracionRealEstimada', 'kwh')
df_capacity_std = transform_dataset(df_capacity, 'Fecha', 'Capacidad acumulada [MW]', 'mwh')
df_demand_std = transform_dataset(df_demand, 'FechaHora', 'Valor', 'kwh')

In [ ]:
df_generation_daily = sum_daily(df_generation_std)
df_capacity_daily = sum_daily(df_capacity_std)
df_demand_daily = sum_daily(df_demand_std)

In [ ]:
df_demand_daily.describe()

In [ ]:
# seaborn is already installed from requirements.txt

## EDA

In [ ]:
df_pred = df_demand_daily.copy()
# Rolling mean simulation for illustration
df_pred['Prediction'] = df_pred['Valor'].rolling(7, min_periods=1).mean()
df_pred['Residual'] = df_pred['Valor'] - df_pred['Prediction']

plt.figure(figsize=(15,5))
plt.plot(df_pred['Fecha'], df_pred['Residual'], color='blue', alpha=0.6)
plt.axhline(df_pred['Residual'].mean(), color='green', linestyle='--', label='Mean Residual')
plt.title('Daily demand residual (Actual value - prediction)')
plt.xlabel('Date')
plt.ylabel('Residual (GWh)')
plt.legend()
plt.tight_layout()
plt.show()

## Code to investigate outlier dates

In [ ]:

# Outlier threshold definition
Q1 = df_demand_daily['Valor'].quantile(0.25)
Q3 = df_demand_daily['Valor'].quantile(0.75)
IQR = Q3 - Q1
upper_thresh = Q3 + 1.5 * IQR

# Select extreme positive outliers
high_outliers = df_demand_daily[df_demand_daily['Valor'] > upper_thresh]

# Save to CSV file
high_outliers[['Fecha', 'Valor']].to_csv('../results/outliers_demanda_altos.csv', index=False)

# Plot the full series and highlight outliers
plt.figure(figsize=(15,5))
plt.plot(df_demand_daily['Fecha'], df_demand_daily['Valor'], color='gray', alpha=0.5, label='Daily demand')
plt.scatter(high_outliers['Fecha'], high_outliers['Valor'], color='red', label='High outliers')
plt.title('Daily demand with highlighted positive outliers')
plt.xlabel('Date')
plt.ylabel('Demand (GWh)')
plt.legend()
plt.tight_layout()
plt.show()

## Demand dataframe outlier cleanup

In [ ]:
mean_value = df_demand_daily['Valor'].mean()

# Define extreme multiplier (e.g.: greater than 1.3 times the mean)
extreme_multiple = 1.3 
extreme_threshold = mean_value * extreme_multiple

# Detect extremes
extremes_mask = df_demand_daily['Valor'] > extreme_threshold

# Create copy for transformations
df_demand_clean = df_demand_daily.copy()

# Replace extreme values with the daily mean
df_demand_clean.loc[extremes_mask, 'Valor'] = mean_value

print(f"Extreme outliers removed/replaced ({extremes_mask.sum()} days).")
print("Clean dataset ready for modeling.")
print("Dates and values of replaced extremes saved.")

## Demand and generation models

In [ ]:
# Demand model
model_demand, forecast_demand, df_cv_demand, df_p_demand = train_predict_validate_prophet(
    df_demand_clean,
    'Demand',
    end_date="2025-10-31"
)

# Generation model
model_generation, forecast_generation, df_cv_generation, df_p_generation = train_predict_validate_prophet(
    df_generation_daily,
    'Generation',
    end_date="2025-10-31"
)

## Model evaluation

The two models created — demand and generation with Prophet — can be evaluated as follows for analytical use:

***

### Demand model

**Evaluation:**  
- **Suitable for exploratory analysis, trend monitoring, and baseline scenarios.**
- **Effectiveness:**  
  - After cleaning extreme outliers, the model shows acceptable performance, with a positive R² (>0.21), low relative errors (MAPE and SMAPE <8%) and reasonable absolute errors (MAE and RMSE).
  - Useful for projections, hypothesis validation, and planning, but it is recommended to complement with exogenous variables or more advanced methods if a high level of reliability or detailed segmentation is required[1][2].
- **Limitations:**  
  - The available history is short, so long-term patterns may be underrepresented.
  - Random events or sudden future changes may not be fully anticipated.

***

### Generation model

**Evaluation:**  
- **Highly robust and strongly recommended for predictive and comparative analysis.**
- **Effectiveness:**  
  - The model retains a very high cross-validated R² (~0.77), low error dispersion, and excellent visual fit in the actual vs. predicted comparison.
  - Reliable for long-term reports, benchmarking, and energy policy decisions.
- **Limitations:**  
  - Like all Prophet models, it depends on the future behaving similarly to the past (without exogenous events or drastic disruptions). If structural changes are expected (new sources or systemic outages), adding regressors or additional modeling layers is recommended.

***

## Evaluation summary

| Variable   | Analytical Use           | Robustness | Recommendation Level |
|------------|--------------------------|------------|----------------------|
| Demand     | Exploration, trend       | Acceptable | Good                 |
| Generation | Prediction and analysis  | High       | Very Good            |

Both models are useful for analysis and decision-making, but the generation model offers more robust and stable results; the demand model is satisfactory for baseline scenarios and operational monitoring after data quality improvement[1][2].

Sources
[1] prophet: Automatic Forecasting Procedure https://community.r-multiverse.org/prophet/prophet.pdf
[2] Time Series Forecasting using Facebook Prophet https://www.nileshdalvi.com/blog/time-series-prophet/

## Save models as .joblib

In [ ]:
# Save models
joblib.dump(model_demand, '../models/modelo_prophet_demanda.joblib')
joblib.dump(model_generation, '../models/modelo_prophet_generacion.joblib')

# To load the model in any script or Streamlit app:
# model_demand = joblib.load('modelo_prophet_demanda.joblib')
# model_generation = joblib.load('modelo_prophet_generacion.joblib')

## Generation vs. Demand comparison through 2030

In [ ]:
import matplotlib.pyplot as plt

# Merge both forecasts on the same date range
df_comparative = forecast_demand[['ds', 'yhat']].rename(columns={'yhat': 'Demanda'})
df_comparative = df_comparative.merge(
    forecast_generation[['ds', 'yhat']].rename(columns={'yhat': 'Generacion'}),
    on='ds', how='inner'
)

# Combined chart
plt.figure(figsize=(18, 6))
plt.plot(df_comparative['ds'], df_comparative['Demanda'], color='orange', label='Predicted Demand')
plt.plot(df_comparative['ds'], df_comparative['Generacion'], color='blue', label='Predicted Generation')
plt.fill_between(df_comparative['ds'], df_comparative['Demanda'], df_comparative['Generacion'],
                 where=(df_comparative['Demanda'] > df_comparative['Generacion']),
                 color='red', alpha=0.2, label='Crisis risk (demand > generation)')
plt.title('Predicted electrical demand vs. generation')
plt.xlabel('Date')
plt.ylabel('Daily GWh')
plt.legend()
plt.tight_layout()
plt.show()

Save projected crisis dates

In [ ]:
import pandas as pd

df_risk = df_comparative[df_comparative['Demanda'] > df_comparative['Generacion']]

# Save to CSV for analysis
df_risk[['ds', 'Demanda', 'Generacion']].to_csv('../results/fechas_riesgo_crisis_energetica.csv', index=False)

print(f"{len(df_risk)} probable energy risk dates identified.")
print("File saved as 'fechas_riesgo_crisis_energetica.csv'.")

## Validation of conclusions
- According to UPME, electrical demand in Colombia will grow on average 2.38% annually until 2038, putting pressure on existing infrastructure and potentially generating a structural energy deficit from 2027 if new generation investments are not made.
- To date, Colombia should add between 3,000 and 4,000 MW of firm capacity per year to meet demand growth, but currently only about 30% of that goal is being achieved.
- The growth potential of renewable energies such as solar and wind is significant. In the last three years, solar capacity in commercial operation has grown by up to 187%, driven by national policies and the Plan Nacional de Desarrollo (PND).
- Over 4,500 MW have already been awarded in non-conventional renewable energy sources (FNCER).
- The country's official goal is to double installed renewable energy capacity by 2030 and reach at least 6 GW by 2026.
- However, structural difficulties exist, including delays in environmental licensing, prior consultations, and limitations in transmission infrastructure, which slow the entry of new projects, especially wind and solar.
## Renewable generation growth variation factors
- The actual growth of installed renewable capacity depends on factors such as:
- Regulatory and administrative agility in project approval.
- Investment in electrical transmission infrastructure and storage systems.
- Mass adoption of distributed generation (residential and industrial solar panels).
- Stability and attractiveness of power purchase agreement (PPA) schemes.
- Incentive policies for self-generation and energy communities.
- In contexts of high renewable penetration, energy production may not coincide with daily consumption peaks, requiring efficient surplus management (batteries, storage systems).

## Report on projections and growth factors for electrical generation in Colombia (2025)
		
Electrical demand in Colombia will increase on average 2.38% annually in the coming years, putting pressure on current infrastructure and requiring investments to avoid a deficit from 2027. Although installed renewable energy capacity (solar and wind) is growing rapidly and doubling targets exist for 2030, the actual pace of growth depends on regulatory, infrastructure, and financial factors. Delays in environmental licensing, transmission limitations, and the need to facilitate self-generation and energy communities may slow the expansion. While Colombia has made progress in renewables (over 4,500 MW awarded and 14% of the energy mix already renewable), maintaining energy sustainability will require continued investment promotion, avoiding bottlenecks, and strengthening storage management and long-term contracts to mitigate price and demand volatility.

Therefore, demand and generation projection models must consider the uncertainties and variability of renewable energy growth, especially in the expansion of solar panels and wind farms, as well as the pace of adoption and socioeconomic factors.

## Sources:
- UPME: Official demand and generation projections.
- Atlas Renewable Energy: Perspectives and trends of the Colombian energy sector.
- El Colombiano: Interview and analysis on growth and deficit in the electrical sector.
- Caracol Radio: Report on projected energy deficit for 2027.
- SITTCA: Renewable momentum analysis and recent figures.
- DNP: Renewable energy reports and projections.
- El Universal: Article on risks of energy shortfall for 2026.
- SER Colombia: Official document on expansion of non-conventional renewable energy sources (FNCER).
- Invest in Colombia: Data on renewable project awards and growth.
- SEI (Stockholm Environment Institute): Studies on solar, wind, and energy communities in Colombia.
- Climatetracker Latinoamérica: Analysis of challenges and opportunities for renewables against energy shortfall.